### EDA and discovery


In [1]:
import pandas as pd
import os

data_dir = 'planets-dataset/planet/planet'
df = pd.read_csv(os.path.join(data_dir, 'train_classes.csv'))
df['filename'] = df['image_name'] + '.jpg'

tags_split = df['tags'].str.get_dummies(sep=' ')
df = pd.concat([df, tags_split], axis=1)

print(df.shape)
print(df.columns.duplicated().sum())  # must print 0 before continuing

(40479, 20)
0


In [2]:
import os

print(os.listdir('planets-dataset/planet/planet'))

['sample_submission.csv', 'test-jpg', 'train-jpg', 'train_classes.csv']


In [3]:
#image folder
image_dir = os.path.join(data_dir, 'train-jpg')
print(os.listdir(image_dir)[:5])

['train_0.jpg', 'train_1.jpg', 'train_10.jpg', 'train_100.jpg', 'train_1000.jpg']


In [25]:
tag_counts = tags_split.sum().sort_values(ascending=False)
print(tag_counts)

primary              37513
clear                28431
agriculture          12315
road                  8071
water                 7411
partly_cloudy         7261
cultivation           4477
habitation            3660
haze                  2697
cloudy                2089
bare_ground            862
selective_logging      340
artisinal_mine         339
blooming               332
slash_burn             209
conventional_mine      100
blow_down               98
dtype: int64


In [26]:
cooccurrence = tags_split.T.dot(tags_split)
print(cooccurrence)


                   agriculture  artisinal_mine  bare_ground  blooming  \
agriculture              12315              38          225        32   
artisinal_mine              38             339           40         0   
bare_ground                225              40          862         3   
blooming                    32               0            3       332   
blow_down                   22               0            4         1   
clear                     9150             307          747       311   
cloudy                       0               0            0         0   
conventional_mine           24               4           10         0   
cultivation               3377              18           89        35   
habitation                2737              29          163         4   
haze                       672               5           41         4   
partly_cloudy             2493              27           74        17   
primary                  11972             324     

In [ ]:
print(df.columns.tolist())
print(df.columns[df.columns.duplicated()])

In [4]:
#tagging priority: resource_extraction > agricultural_clearing > infrastructure_other > undisturbed

import numpy as np

extraction_tags = ['selective_logging', 'artisinal_mine', 'conventional_mine']
agriculture_tags = ['agriculture', 'cultivation', 'slash_burn']
infrastructure_tags = ['road', 'habitation', 'blow_down']

conditions = [
    df[extraction_tags].sum(axis=1) > 0,
    df[agriculture_tags].sum(axis=1) > 0,
    df[infrastructure_tags].sum(axis=1) > 0,
    df['primary'] == 1,
]
choices = ['resource_extraction', 'agricultural_clearing', 'infrastructure_other', 'undisturbed']

df['label'] = np.select(conditions, choices, default='unlabeled')
print(df['label'].value_counts())

label
undisturbed              22038
agricultural_clearing    13300
unlabeled                 2472
infrastructure_other      1900
resource_extraction        769
Name: count, dtype: int64


In [5]:
#remove unlabled rows
df_clean = df[df['label'] != 'unlabeled'].copy()
print(len(df_clean))

38007


In [8]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

classes = np.array(df_clean['label'].unique())
weights = compute_class_weight('balanced', classes=classes, y=df_clean['label'])
class_weight_dict = dict(zip(classes, weights))
print(class_weight_dict)

# convert to tensor in the order your model's output layer expects
class_names = ['undisturbed', 'agricultural_clearing', 'infrastructure_other', 'resource_extraction']
weight_tensor = torch.tensor([class_weight_dict[c] for c in class_names], dtype=torch.float32)

{'undisturbed': np.float64(0.43115300843996734), 'agricultural_clearing': np.float64(0.7144172932330827), 'infrastructure_other': np.float64(5.000921052631579), 'resource_extraction': np.float64(12.355981794538362)}


In [9]:
class_names = ['undisturbed', 'agricultural_clearing', 'infrastructure_other', 'resource_extraction']
weight_tensor = torch.tensor([class_weight_dict[c] for c in class_names], dtype=torch.float32)
print(weight_tensor)

tensor([ 0.4312,  0.7144,  5.0009, 12.3560])


In [10]:
label_to_idx = {name: i for i, name in enumerate(class_names)}
df_clean['label_idx'] = df_clean['label'].map(label_to_idx)

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    stratify=df_clean['label'], 
    random_state=42
)
print(train_df['label'].value_counts(normalize=True))
print(test_df['label'].value_counts(normalize=True))

label
undisturbed              0.579839
agricultural_clearing    0.349942
infrastructure_other     0.049992
resource_extraction      0.020227
Name: proportion, dtype: float64
label
undisturbed              0.579847
agricultural_clearing    0.349908
infrastructure_other     0.049987
resource_extraction      0.020258
Name: proportion, dtype: float64


In [12]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class ForestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        label = row['label_idx']
        if self.transform:
            image = self.transform(image)
        return image, label